In [1]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
import torch
import napari
import os
from skimage import io, draw, measure
import numpy as np
import pandas as pd
import dask.array as da
from ome_zarr.image import NgffMultiscales
from ome_zarr.scene import NgffScene

In [2]:
model = YOLO(r"train8\weights\best.pt")

In [3]:
viewer = napari.Viewer()

In [7]:
scene = NgffScene.from_ome_zarr(r"C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\10x\J\10\scene.ome.zarr")
scene.images

[NgffMultiscales(method=<Methods.RESIZE: 'resize'>),
 NgffMultiscales(method=<Methods.RESIZE: 'resize'>),
 NgffMultiscales(method=<Methods.RESIZE: 'resize'>),
 NgffMultiscales(method=<Methods.RESIZE: 'resize'>)]

In [8]:
def yolo_box2rect(box):
	y_center, x_center, height, width = box
	x_min = int(x_center - width / 2)
	x_max = int(x_center + width / 2)
	y_min = int(y_center - height / 2)
	y_max = int(y_center + height / 2)

	p0 = (x_min, y_min)
	p1 = (x_max, y_min)
	p2 = (x_max, y_max)
	p3 = (x_min, y_max)

	rect = [p0, p1, p2, p3]
	return rect

In [9]:
def sliding_window_inference(image, model, window_size=(640, 640), overlap=0.2):
	height, width = image.shape[:2]
	win_h, win_w = window_size
	stride_h = int(win_h * (1 - overlap))
	stride_w = int(win_w * (1 - overlap))

	windows = {}

	# Generate all window positions
	y_positions = list(range(0, height - win_h + 1, stride_h))
	x_positions = list(range(0, width - win_w + 1, stride_w))

	# Ensure the far edges are covered by adding a final window at the edge
	if y_positions[-1] + win_h < height:
		y_positions.append(height - win_h)
	if x_positions[-1] + win_w < width:
		x_positions.append(width - win_w)

	# Slide windows across image
	for y_offset in y_positions:
		for x_offset in x_positions:
			window = image[y_offset:y_offset+win_h, x_offset:x_offset+win_w]
			window_norm = (window - window.min()) / (window.max() - window.min() + 1e-8) * 255
			windows[(y_offset, x_offset)] = window_norm

	predictions = model(list(windows.values()))
	df = pd.DataFrame()
	for window, prediction in zip(windows.keys(), predictions):
		y_offset, x_offset = window
		for box, conf in zip(prediction.boxes.xywh, prediction.boxes.conf):
			rectangle = yolo_box2rect(box)
			# Shift rectangle coordinates by window offset
			rectangle_shifted = [(y + y_offset, x + x_offset) for (y, x) in rectangle]
			_df = pd.DataFrame({
				'x0': [rectangle_shifted[0][1]],
				'y0': [rectangle_shifted[0][0]],
				'x1': [rectangle_shifted[2][1]],
				'y1': [rectangle_shifted[2][0]],
				'confidence': [conf.item()]
			})
			df = pd.concat([df, _df], ignore_index=True)

	return df

In [10]:
def extract_features(image: np.ndarray, boxes: pd.DataFrame) -> pd.DataFrame:
	"""Basic feature extraction using skimage.measure.regionprops_table."""

	features = pd.DataFrame()

	for _, box in boxes.iterrows():
		x0, y0, x1, y1 = int(box['x0']), int(box['y0']), int(box['x1']), int(box['y1'])
		crop = image[y0:y1, x0:x1]

		_features = pd.DataFrame(measure.regionprops_table(
			label_image=np.ones(crop.shape, dtype=int),  # Dummy label image
			intensity_image=crop,
			properties=[
				"area",
				"mean_intensity",
				"max_intensity",
				"min_intensity",
			],
		))
		_features["x0"] = x0
		_features["y0"] = y0
		_features["x1"] = x1
		_features["y1"] = y1
		_features["width"] = x1 - x0
		_features["height"] = y1 - y0
		if "confidence" in box:
			_features["confidence"] = box["confidence"]
		features = pd.concat([features, _features], ignore_index=True)
	# _label = draw.polygon2mask(projection.shape, [p0, p1, p2, p3]).astype(int)

	# label = label.squeeze(-1)  # Remove the channel axis if present



	# roi_feat_table = pd.DataFrame(measure.regionprops_table(
	# 		label_image=label,
	# 		intensity_image=image,
	# 		properties=[
	# 			"label",
	# 			"area",
	# 			"mean_intensity",
	# 			"max_intensity",
	# 			"min_intensity",
	# 		],
	# 	)
	# )
	
	# prediction = model(image)[0]
	# roi_feat_table = pd.DataFrame(columns=[
	# 	"x0", "y0", "x1", "y1",
	# 	"width", "height", "x_center", "y_center",
	# 	"area", "confidence"
	# ])

	# for idx, (box, conf) in enumerate(zip(prediction.boxes.xywh, prediction.boxes.conf)):
	# 	p0, p1, p2, p3 = yolo_box2bbox(box)
	# 	x_min, y_min = p0
	# 	x_max, y_max = p2

	# 	roi_feat_table.loc[idx, "x0"] = x_min
	# 	roi_feat_table.loc[idx, "y0"] = y_min
	# 	roi_feat_table.loc[idx, "x1"] = x_max
	# 	roi_feat_table.loc[idx, "y1"] = y_max
	# 	roi_feat_table.loc[idx, "width"] = x_max - x_min
	# 	roi_feat_table.loc[idx, "height"] = y_max - y_min
	# 	roi_feat_table.loc[idx, "x_center"] = (x_min + x_max) / 2
	# 	roi_feat_table.loc[idx, "y_center"] = (y_min + y_max) / 2
	# 	roi_feat_table.loc[idx, "area"] = (x_max - x_min) * (y_max - y_min)
	# 	roi_feat_table.loc[idx, "confidence"] = conf.detach().cpu().numpy()

	return features

In [11]:
def remove_duplicate_detections(df, iou_threshold=0.5):
	"""Remove duplicate detections in overlap regions using IoU."""
	if len(df) == 0:
		return df
	
	# Sort by confidence (highest first)
	df = df.sort_values('confidence', ascending=False).reset_index(drop=True)
	keep = []

	boxes_xyxy = df[['x0', 'y0', 'x1', 'y1']].values
	ious = box_iou(torch.tensor(boxes_xyxy), torch.tensor(boxes_xyxy)).numpy()
	np.fill_diagonal(ious, 0)  # Ignore self-comparison
	
	for i, row in df.iterrows():
		keep_this = True
		for kept_idx in keep:
			if ious[i, kept_idx] > iou_threshold:
				keep_this = False
				break
		
		if keep_this:
			keep.append(i)
	
	return df.loc[keep].reset_index(drop=True)

In [12]:
viewer.layers.clear()

for ms in scene.images:
	# retrieve translation and multiscales
	translation = [t for t in scene.metadata.coordinateTransformations if t.input.path == ms.name][0]
	translate = translation.translation[1:]
	scale=list(ms.images[0].scale.values())[1:]

	projection = ms.images[0].data.min(axis=0)
	projection = (projection - projection.min()) / (projection.max() - projection.min()) * 255


	df_rectangles = sliding_window_inference(projection.compute(), model, window_size=(640, 640), overlap=0.2)
	df_rectangles = remove_duplicate_detections(df_rectangles, iou_threshold=0.1)
	df_rectangles = extract_features(projection.compute(), df_rectangles)
	
	rectangles = []
	for _, row in df_rectangles.iterrows():
		x0, y0, x1, y1 = int(row['x0']), int(row['y0']), int(row['x1']), int(row['y1'])
		rectangles.append([(y0, x0), (y0, x1), (y1, x1), (y1, x0)])

	df_rectangles.drop(columns=["x0", "y0", "x1", "y1"], inplace=True)

	_ = viewer.add_image(projection, name=os.path.basename(ms.name), translate=translate, scale=scale)
	_ = viewer.add_shapes(
		rectangles, shape_type="rectangle", edge_color="red",
		face_color="transparent", features=df_rectangles, edge_width=5, translate=translate, scale=scale)



0: 640x640 (no detections), 48.4ms
1: 640x640 1 organoid, 48.4ms
2: 640x640 2 organoids, 48.4ms
3: 640x640 (no detections), 48.4ms
4: 640x640 1 organoid, 48.4ms
5: 640x640 8 organoids, 48.4ms
6: 640x640 (no detections), 48.4ms
7: 640x640 1 organoid, 48.4ms
8: 640x640 1 organoid, 48.4ms
9: 640x640 1 organoid, 48.4ms
10: 640x640 (no detections), 48.4ms
11: 640x640 2 organoids, 48.4ms
12: 640x640 (no detections), 48.4ms
13: 640x640 (no detections), 48.4ms
14: 640x640 (no detections), 48.4ms
15: 640x640 (no detections), 48.4ms
16: 640x640 (no detections), 48.4ms
17: 640x640 1 organoid, 48.4ms
18: 640x640 (no detections), 48.4ms
19: 640x640 (no detections), 48.4ms
20: 640x640 2 organoids, 48.4ms
21: 640x640 1 organoid, 48.4ms
22: 640x640 (no detections), 48.4ms
23: 640x640 2 organoids, 48.4ms
Speed: 2.8ms preprocess, 48.4ms inference, 0.3ms postprocess per image at shape (1, 1, 640, 640)

0: 640x640 3 organoids, 38.8ms
1: 640x640 8 organoids, 38.8ms
2: 640x640 3 organoids, 38.8ms
3: 640x64

C:\Users\johan\Documents\GitHub\biaplotter\src\biaplotter\colormap.py:34: UserWarning: Categorical colormap detected. Setting categorical=True. If the colormap is continuous, set categorical=False explicitly.
  warnings.warn(
C:\Users\johan\Documents\GitHub\biaplotter\src\biaplotter\artists.py:331: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\matplotlib\cbook.py", line 361, in process
    func(*args, **kwargs)
  File "c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\matplotlib\backend_bases.py", line 2956, in mouse_move
    self.set_message(self._mouse_event_to_message(event))
  File "c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\matplotlib\backends\backend_qt.py", line 809, in set_message
    self.locLabel.setText(s)
RuntimeError: wrap

In [35]:
box1 = viewer.layers[-1].data[list(viewer.layers[-1].selected_data)[0]]
box2 = viewer.layers[-1].data[list(viewer.layers[-1].selected_data)[1]]

In [50]:
box1_tensor = torch.tensor([[box1[0, 0], box1[0, 1], box1[2, 0], box1[2, 1]]])
box2_tensor = torch.tensor([[box2[0, 0], box2[0, 1], box2[2, 0], box2[2, 1]]])

In [52]:
box_iou(box1_tensor, box2_tensor)

tensor([[0.8161]])

In [91]:
tile = projection.compute()[1*640:2*640, 1*640:2*640]
prediction = model(tile)[0]
rectangle = yolo_box2rect(prediction.boxes.xywh[0])


0: 640x640 1 organoid, 111.4ms
Speed: 3.4ms preprocess, 111.4ms inference, 0.3ms postprocess per image at shape (1, 1, 640, 640)


In [92]:
viewer.layers.clear()
viewer.add_image(tile)
viewer.add_shapes([rectangle], shape_type="rectangle", edge_color="red", face_color="transparent", edge_width=5)

<Shapes layer 'Shapes' at 0x1e1cf742b70>